# Lezione 2 - ANN, CNN e Transfer Learning su immagini

Notebook didattico su immagini con PyTorch.

Percorso della lezione:

1. Fashion-MNIST in scala di grigi;
2. flatten/unfold dell'immagine per usare una ANN;
3. ANN fully connected;
4. CNN su Fashion-MNIST;
5. immagini RGB con CIFAR-10;
6. CNN su immagini a colori;
7. transfer learning con ResNet18.

La struttura resta coerente con la Lezione 1: split train/validation/test, funzioni di training riutilizzabili, validation loss e test finale.


## 0. Dipendenze

Se il notebook viene eseguito in un ambiente nuovo, installare prima:

```bash
pip install torch torchvision scikit-learn pandas matplotlib numpy
```

I dataset vengono scaricati automaticamente da `torchvision` la prima volta che si esegue il notebook. Anche i pesi pre-addestrati della ResNet possono richiedere un download alla prima esecuzione.


In [1]:
import importlib.util

required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
}

missing = [pip_name for module_name, pip_name in required_packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    print("Librerie mancanti:", ", ".join(missing))
    print("Eseguire in una cella separata:")
    print("%pip install " + " ".join(missing))
else:
    print("Tutte le dipendenze sono disponibili.")

Librerie mancanti: torchvision
Eseguire in una cella separata:
%pip install torchvision


In [ ]:

pip install torchvision

SyntaxError: invalid syntax (3866160270.py, line 1)

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in uso:", device)

KeyboardInterrupt: 

## 1. Fashion-MNIST: che tipo di problema e?

Fashion-MNIST contiene immagini in scala di grigi di capi di abbigliamento, scarpe e borse.

Ogni immagine ha dimensione:

$$28 \times 28$$

Quindi ogni immagine contiene 784 pixel.

Per una ANN completamente connessa dobbiamo trasformare ogni immagine da matrice `28 x 28` a vettore di lunghezza `784`. Questa operazione si chiama **flatten**.

Il problema e di classificazione multiclasse:

- input: immagine di un capo di abbigliamento o accessorio;
- output: una delle 10 classi, per esempio `T-shirt/top`, `Trouser`, `Sneaker`, `Bag`.


## 2. Caricamento del dataset Fashion-MNIST

Useremo `torchvision.datasets.FashionMNIST`.

Per rendere il notebook piu veloce durante una lezione, di default useremo un sottoinsieme del dataset. Per usare tutto Fashion-MNIST basta impostare:

```python
USE_SUBSET = False
```


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_full = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

test_full = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

print("Immagini train originali:", len(train_full))
print("Immagini test originali:", len(test_full))
class_names = train_full.classes
print("Classi:", class_names)

In [ ]:
USE_SUBSET = True
TRAIN_SUBSET_SIZE = 12000
TEST_SUBSET_SIZE = 2000
BATCH_SIZE = 64

if USE_SUBSET:
    generator = torch.Generator().manual_seed(SEED)
    train_indices = torch.randperm(len(train_full), generator=generator)[:TRAIN_SUBSET_SIZE]
    test_indices = torch.randperm(len(test_full), generator=generator)[:TEST_SUBSET_SIZE]
    train_source = Subset(train_full, train_indices)
    test_dataset = Subset(test_full, test_indices)
else:
    train_source = train_full
    test_dataset = test_full

val_size = int(0.2 * len(train_source))
train_size = len(train_source) - val_size

train_dataset, val_dataset = random_split(
    train_source,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

## 3. Visualizzazione di alcune immagini

Prima di costruire la rete, osserviamo qualche esempio del dataset.

Le immagini sono normalizzate per il training. Per visualizzarle in modo piu naturale applichiamo una piccola funzione di de-normalizzazione.

In [ ]:
def denormalize(image_tensor):
    return image_tensor * 0.3530 + 0.2860

images, labels = next(iter(train_loader))

plt.figure(figsize=(9, 4))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    image = denormalize(images[i]).squeeze().numpy()
    plt.imshow(image, cmap="gray")
    plt.title(class_names[labels[i].item()], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Perche serve il flatten?

Una ANN classica lavora con vettori, non direttamente con immagini 2D.

Un'immagine Fashion-MNIST ha forma:

```python
1 x 28 x 28
```

Dove `1` indica il canale colore, perche l'immagine e in scala di grigi.

Prima del primo layer lineare la trasformiamo in:

```python
784
```

Questa rete puo classificare immagini, ma perde la struttura spaziale locale. La CNN, che vedremo nella Parte B, serve proprio a sfruttare meglio questa struttura.

In [ ]:
# Mini-demo: da immagine 2D a vettore per una ANN
images, labels = next(iter(train_loader))
print("Batch originale:", images.shape)  # batch, canali, altezza, larghezza

flatten = nn.Flatten()
images_flat = flatten(images)
print("Batch dopo flatten:", images_flat.shape)  # batch, 784
print("Una immagine 28 x 28 diventa un vettore con", images_flat.shape[1], "valori")

# Parte A - ANN con hidden layer

Costruiamo direttamente una ANN con hidden layer.

Architettura:

- flatten: da `1 x 28 x 28` a `784` valori;
- hidden layer 1: 128 neuroni + ReLU;
- hidden layer 2: 64 neuroni + ReLU;
- output layer: 10 neuroni, uno per classe Fashion-MNIST.

Per la classificazione multiclasse useremo `CrossEntropyLoss`.


In [ ]:
class FashionMNISTMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.network(x)

model = FashionMNISTMLP().to(device)
print(model)
print("Numero di parametri:", sum(p.numel() for p in model.parameters()))

## Funzioni comuni per training, validation e test

Queste funzioni sono pensate per essere riutilizzate in entrambi i notebook.

L'unica differenza tra classificazione binaria e multiclasse viene gestita con il parametro `task`:

- `task="binary"`: output a 1 neurone, `BCEWithLogitsLoss`, sigmoide e soglia 0.5;
- `task="multiclass"`: output a N classi, `CrossEntropyLoss`, softmax e argmax.

In [ ]:
def run_epoch(model, data_loader, criterion, optimizer=None, task="multiclass", threshold=0.5):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, targets in data_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(inputs)
            loss = criterion(logits, targets)

            if is_training:
                loss.backward()
                optimizer.step()

        if task == "binary":
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= threshold).float()
            correct = (predictions == targets.view_as(predictions)).sum().item()
        elif task == "multiclass":
            predictions = logits.argmax(dim=1)
            correct = (predictions == targets).sum().item()
        else:
            raise ValueError("task deve essere 'binary' oppure 'multiclass'")

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_correct += correct
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


def train_model(
    model,
    train_loader,
    val_loader,
    criterion=None,
    task="multiclass",
    epochs=10,
    lr=0.001,
    weight_decay=0.0,
    threshold=0.5,
):
    model = model.to(device)
    if criterion is None:
        criterion = nn.BCEWithLogitsLoss() if task == "binary" else nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
    }

    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer=optimizer,
            task=task,
            threshold=threshold,
        )
        val_loss, val_acc = run_epoch(
            model,
            val_loader,
            criterion,
            optimizer=None,
            task=task,
            threshold=threshold,
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoca {epoch + 1:02d}/{epochs} - "
            f"train loss: {train_loss:.4f} - val loss: {val_loss:.4f} - "
            f"train acc: {train_acc:.3f} - val acc: {val_acc:.3f}"
        )

    return history


def evaluate_model(model, data_loader, task="multiclass", threshold=0.5):
    model.eval()
    all_targets = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            logits = model(inputs)

            if task == "binary":
                probabilities = torch.sigmoid(logits).cpu().numpy().reshape(-1)
                predictions = (probabilities >= threshold).astype(int)
                targets_np = targets.cpu().numpy().reshape(-1).astype(int)
            elif task == "multiclass":
                probabilities = torch.softmax(logits, dim=1).cpu().numpy()
                predictions = probabilities.argmax(axis=1)
                targets_np = targets.cpu().numpy().reshape(-1).astype(int)
            else:
                raise ValueError("task deve essere 'binary' oppure 'multiclass'")

            all_targets.extend(targets_np)
            all_predictions.extend(predictions)
            all_probabilities.extend(probabilities)

    y_true = np.array(all_targets)
    y_pred = np.array(all_predictions)
    probabilities = np.array(all_probabilities)
    accuracy = accuracy_score(y_true, y_pred)
    return accuracy, y_true, y_pred, probabilities


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_loss"], label="Training loss")
    axes[0].plot(history["val_loss"], label="Validation loss")
    axes[0].set_title(title + " - loss")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    axes[1].plot(history["train_accuracy"], label="Training accuracy")
    axes[1].plot(history["val_accuracy"], label="Validation accuracy")
    axes[1].set_title(title + " - accuracy")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


def plot_loss(history, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_loss"], label="Training loss")
    plt.plot(history["val_loss"], label="Validation loss")
    plt.title(title)
    plt.xlabel("Epoca")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


def plot_accuracy(history, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_accuracy"], label="Training accuracy")
    plt.plot(history["val_accuracy"], label="Validation accuracy")
    plt.title(title)
    plt.xlabel("Epoca")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


def plot_confusion_matrix(cm, title, class_names, cmap="Blues"):
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap=cmap)
    plt.title(title)
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)
    plt.xlabel("Predetto")
    plt.ylabel("Reale")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > cm.max() * 0.6 else "black"
            plt.text(j, i, cm[i, j], ha="center", va="center", color=color, fontsize=8)

    plt.colorbar()
    plt.tight_layout()
    plt.show()

In [ ]:
history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.001,
    weight_decay=1e-4,
)

In [ ]:
plot_loss(history, "Loss per epoca - ANN su Fashion-MNIST")
plot_accuracy(history, "Accuracy per epoca - ANN su Fashion-MNIST")

## 6. Valutazione sul test set

Il test set viene usato solo alla fine, dopo aver scelto il modello e osservato la validation loss.

In [ ]:
test_accuracy, y_test_true, y_test_pred, y_test_proba = evaluate_model(
    model,
    test_loader,
    task="multiclass",
)

ann_test_accuracy = test_accuracy
ann_y_test_true = y_test_true
ann_y_test_pred = y_test_pred
ann_y_test_proba = y_test_proba

print(f"Accuracy test: {test_accuracy:.3f}")
print("\nClassification report sul test set:")
print(classification_report(y_test_true, y_test_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_test_true, y_test_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion matrix - ANN su Fashion-MNIST")
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = "white" if cm[i, j] > cm.max() * 0.6 else "black"
        plt.text(j, i, cm[i, j], ha="center", va="center", color=color, fontsize=8)

plt.colorbar()
plt.show()

## 7. Predizioni su immagini di esempio

Visualizziamo alcune immagini del test set con la classe reale, la classe prevista e la probabilita assegnata dal modello.

In [ ]:
def show_predictions(model, data_loader, n=12):
    model.eval()
    images, labels = next(iter(data_loader))
    images_device = images.to(device)

    with torch.no_grad():
        logits = model(images_device)
        probabilities = torch.softmax(logits, dim=1).cpu()
        predictions = probabilities.argmax(dim=1)

    plt.figure(figsize=(10, 5))
    for i in range(n):
        plt.subplot(2, 6, i + 1)
        image = denormalize(images[i]).squeeze().numpy()
        confidence = probabilities[i, predictions[i]].item()
        plt.imshow(image, cmap="gray")
        plt.title(f"R:{class_names[labels[i].item()]}\nP:{class_names[predictions[i].item()]} ({confidence:.2f})", fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_predictions(model, test_loader, n=12)

## 8. Come leggere training loss e validation loss

- Se training loss e validation loss scendono insieme, il modello sta imparando in modo ragionevole.
- Se training loss scende ma validation loss sale, il modello potrebbe stare andando in overfitting.
- Se entrambe restano alte, la rete potrebbe essere troppo semplice, le epoche troppo poche o il learning rate non adatto.
- Con Fashion-MNIST, una ANN funziona bene, ma non sfrutta in modo esplicito la struttura spaziale dell'immagine. Per questo nella Parte B useremo una CNN.

# Parte B - CNN su Fashion-MNIST

Ora implementiamo una **Convolutional Neural Network (CNN)**.

La differenza principale rispetto alla ANN e che la CNN non appiattisce subito l'immagine. Prima usa filtri convoluzionali per imparare pattern locali, come bordi, forme e texture.

Architettura:

1. `Conv2d`: da 1 canale a 16 mappe di feature;
2. ReLU;
3. `MaxPool2d`: riduce la dimensione da `28 x 28` a `14 x 14`;
4. `Conv2d`: da 16 a 32 mappe di feature;
5. ReLU;
6. `MaxPool2d`: riduce da `14 x 14` a `7 x 7`;
7. flatten;
8. layer fully connected finale.

In [ ]:
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = FashionMNISTCNN().to(device)
print(cnn_model)
print("Numero di parametri:", sum(p.numel() for p in cnn_model.parameters()))

## 11. Training della CNN

Usiamo la stessa funzione di training della ANN. Anche qui salviamo training loss, validation loss, training accuracy e validation accuracy.

In [ ]:
history_cnn = train_model(
    cnn_model,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.001,
    weight_decay=1e-4,
)

In [ ]:
plot_loss(history_cnn, "Loss per epoca - CNN su Fashion-MNIST")
plot_accuracy(history_cnn, "Accuracy per epoca - CNN su Fashion-MNIST")

## 12. Valutazione della CNN sul test set

Ora valutiamo la CNN sullo stesso test set usato per la ANN.

In [ ]:
cnn_test_accuracy, cnn_y_test_true, cnn_y_test_pred, cnn_y_test_proba = evaluate_model(cnn_model, test_loader, task="multiclass")
print(f"Accuracy test CNN: {cnn_test_accuracy:.3f}")
print("\nClassification report CNN sul test set:")
print(classification_report(cnn_y_test_true, cnn_y_test_pred, target_names=class_names, digits=3))

In [ ]:
cm_cnn = confusion_matrix(cnn_y_test_true, cnn_y_test_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm_cnn, cmap="Greens")
plt.title("Confusion matrix - CNN su Fashion-MNIST")
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)

for i in range(cm_cnn.shape[0]):
    for j in range(cm_cnn.shape[1]):
        color = "white" if cm_cnn[i, j] > cm_cnn.max() * 0.6 else "black"
        plt.text(j, i, cm_cnn[i, j], ha="center", va="center", color=color, fontsize=8)

plt.colorbar()
plt.show()

## 13. Predizioni della CNN su immagini di esempio

Usiamo la stessa funzione di visualizzazione, ma questa volta passiamo il modello CNN.

In [ ]:
show_predictions(cnn_model, test_loader, n=12)

## 14. Confronto ANN vs CNN

Confrontiamo i due modelli sulla stessa divisione train/validation/test.

In genere la CNN dovrebbe comportarsi meglio perche usa la struttura spaziale dell'immagine, mentre la ANN lavora sul vettore appiattito di pixel.

In [ ]:
comparison = pd.DataFrame({
    "modello": ["ANN hidden layer", "CNN"],
    "final_train_loss": [history["train_loss"][-1], history_cnn["train_loss"][-1]],
    "final_val_loss": [history["val_loss"][-1], history_cnn["val_loss"][-1]],
    "final_train_accuracy": [history["train_accuracy"][-1], history_cnn["train_accuracy"][-1]],
    "final_val_accuracy": [history["val_accuracy"][-1], history_cnn["val_accuracy"][-1]],
    "test_accuracy": [ann_test_accuracy, cnn_test_accuracy],
    "parametri": [
        sum(p.numel() for p in model.parameters()),
        sum(p.numel() for p in cnn_model.parameters()),
    ],
})

comparison

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["val_loss"], label="ANN - validation loss")
plt.plot(history_cnn["val_loss"], label="CNN - validation loss")
plt.title("Confronto validation loss - ANN vs CNN")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["val_accuracy"], label="ANN - validation accuracy")
plt.plot(history_cnn["val_accuracy"], label="CNN - validation accuracy")
plt.title("Confronto validation accuracy - ANN vs CNN")
plt.xlabel("Epoca")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 15. Perche la CNN e piu adatta alle immagini?

Una ANN vede l'immagine come una lista di 784 numeri. In questo modo perde l'informazione di vicinanza tra pixel.

Una CNN, invece, conserva la struttura 2D e usa filtri locali. Questo le permette di imparare pattern come bordi, curve e texture, che sono molto utili per riconoscere oggetti visivi.

# Parte C - Immagini a colori con CIFAR-10

Finora abbiamo lavorato con Fashion-MNIST, che contiene immagini in scala di grigi con forma:

```python
1 x 28 x 28
```

Ora passiamo a **CIFAR-10**, un dataset di immagini a colori. Ogni immagine ha forma:

```python
3 x 32 x 32
```

Il `3` indica i tre canali RGB: rosso, verde e blu.

CIFAR-10 ha 10 classi: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
cifar_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

cifar_train_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=cifar_train_transform,
)

cifar_test_full = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=cifar_test_transform,
)

cifar_class_names = cifar_train_full.classes
print("Immagini train CIFAR-10:", len(cifar_train_full))
print("Immagini test CIFAR-10:", len(cifar_test_full))
print("Classi:", cifar_class_names)

In [ ]:
USE_CIFAR_SUBSET = True
CIFAR_TRAIN_SUBSET_SIZE = 15000
CIFAR_TEST_SUBSET_SIZE = 3000
CIFAR_BATCH_SIZE = 64

if USE_CIFAR_SUBSET:
    generator = torch.Generator().manual_seed(SEED)
    cifar_train_indices = torch.randperm(len(cifar_train_full), generator=generator)[:CIFAR_TRAIN_SUBSET_SIZE]
    cifar_test_indices = torch.randperm(len(cifar_test_full), generator=generator)[:CIFAR_TEST_SUBSET_SIZE]
    cifar_train_source = Subset(cifar_train_full, cifar_train_indices)
    cifar_test_dataset = Subset(cifar_test_full, cifar_test_indices)
else:
    cifar_train_source = cifar_train_full
    cifar_test_dataset = cifar_test_full

cifar_val_size = int(0.2 * len(cifar_train_source))
cifar_train_size = len(cifar_train_source) - cifar_val_size

cifar_train_dataset, cifar_val_dataset = random_split(
    cifar_train_source,
    [cifar_train_size, cifar_val_size],
    generator=torch.Generator().manual_seed(SEED),
)

cifar_train_loader = DataLoader(cifar_train_dataset, batch_size=CIFAR_BATCH_SIZE, shuffle=True)
cifar_val_loader = DataLoader(cifar_val_dataset, batch_size=CIFAR_BATCH_SIZE, shuffle=False)
cifar_test_loader = DataLoader(cifar_test_dataset, batch_size=CIFAR_BATCH_SIZE, shuffle=False)

print("CIFAR train:", len(cifar_train_dataset))
print("CIFAR validation:", len(cifar_val_dataset))
print("CIFAR test:", len(cifar_test_dataset))

## 18. Visualizzazione di immagini RGB

Per visualizzare immagini normalizzate dobbiamo invertire la normalizzazione, canale per canale.

In [ ]:
def denormalize_cifar(image_tensor):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)

cifar_images, cifar_labels = next(iter(cifar_train_loader))

plt.figure(figsize=(10, 5))
for i in range(12):
    plt.subplot(2, 6, i + 1)
    image = denormalize_cifar(cifar_images[i]).permute(1, 2, 0).numpy()
    plt.imshow(image)
    plt.title(cifar_class_names[cifar_labels[i].item()], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 19. CNN semplice per CIFAR-10

Per immagini RGB il primo layer convoluzionale deve avere `in_channels=3`, perche l'input ha tre canali colore.

Questa CNN e volutamente semplice, cosi resta adatta a una lezione introduttiva.

In [ ]:
class CIFAR10SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cifar_cnn_model = CIFAR10SmallCNN().to(device)
print(cifar_cnn_model)
print("Numero di parametri:", sum(p.numel() for p in cifar_cnn_model.parameters()))

In [ ]:
history_cifar_cnn = train_model(
    cifar_cnn_model,
    cifar_train_loader,
    cifar_val_loader,
    epochs=10,
    lr=0.001,
    weight_decay=1e-4,
)

In [ ]:
plot_loss(history_cifar_cnn, "Loss per epoca - CNN su CIFAR-10")
plot_accuracy(history_cifar_cnn, "Accuracy per epoca - CNN su CIFAR-10")

In [ ]:
cifar_cnn_test_accuracy, cifar_y_test_true, cifar_y_test_pred, cifar_y_test_proba = evaluate_model(cifar_cnn_model, cifar_test_loader, task="multiclass")
print(f"Accuracy test CNN CIFAR-10: {cifar_cnn_test_accuracy:.3f}")
print("\nClassification report CNN CIFAR-10:")
print(classification_report(cifar_y_test_true, cifar_y_test_pred, target_names=cifar_class_names, digits=3))

In [ ]:
cm_cifar = confusion_matrix(cifar_y_test_true, cifar_y_test_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm_cifar, cmap="Purples")
plt.title("Confusion matrix - CNN su CIFAR-10")
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.xticks(range(10), cifar_class_names, rotation=45, ha="right")
plt.yticks(range(10), cifar_class_names)

for i in range(cm_cifar.shape[0]):
    for j in range(cm_cifar.shape[1]):
        color = "white" if cm_cifar[i, j] > cm_cifar.max() * 0.6 else "black"
        plt.text(j, i, cm_cifar[i, j], ha="center", va="center", color=color, fontsize=8)

plt.colorbar()
plt.show()

In [ ]:
def show_cifar_predictions(model, data_loader, n=12):
    model.eval()
    images, labels = next(iter(data_loader))
    images_device = images.to(device)

    with torch.no_grad():
        logits = model(images_device)
        probabilities = torch.softmax(logits, dim=1).cpu()
        predictions = probabilities.argmax(dim=1)

    plt.figure(figsize=(10, 5))
    for i in range(n):
        plt.subplot(2, 6, i + 1)
        image = denormalize_cifar(images[i]).permute(1, 2, 0).numpy()
        confidence = probabilities[i, predictions[i]].item()
        plt.imshow(image)
        plt.title(
            f"R:{cifar_class_names[labels[i].item()]}\nP:{cifar_class_names[predictions[i].item()]} ({confidence:.2f})",
            fontsize=8,
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_cifar_predictions(cifar_cnn_model, cifar_test_loader, n=12)

# Parte D - Transfer learning con ResNet

Nel transfer learning partiamo da un modello gia addestrato su un grande dataset, tipicamente ImageNet, e lo adattiamo al nostro problema.

Idea:

1. carichiamo una ResNet pre-addestrata;
2. congeliamo i layer convoluzionali iniziali;
3. sostituiamo l'ultimo layer fully connected con un layer da 10 classi;
4. alleniamo solo il classificatore finale, almeno nella prima fase.

Questo approccio e utile quando abbiamo pochi dati o vogliamo partire da feature visive gia apprese.

## 20. Dataset CIFAR-10 adattato a ResNet

ResNet pre-addestrata su ImageNet si aspetta immagini RGB normalizzate con media e deviazione standard di ImageNet. Inoltre, per compatibilita didattica, ridimensioniamo le immagini a `224 x 224`.

Questa parte puo essere piu lenta della CNN precedente. Per la lezione manteniamo un subset piccolo di default.

In [ ]:
from torchvision import models
from torchvision.models import ResNet18_Weights

resnet_weights = ResNet18_Weights.DEFAULT
imagenet_mean = resnet_weights.transforms().mean
imagenet_std = resnet_weights.transforms().std

transfer_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

transfer_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

transfer_train_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transfer_train_transform,
)

transfer_test_full = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transfer_test_transform,
)

In [ ]:
USE_TRANSFER_SUBSET = True
TRANSFER_TRAIN_SUBSET_SIZE = 4000
TRANSFER_TEST_SUBSET_SIZE = 1000
TRANSFER_BATCH_SIZE = 32

if USE_TRANSFER_SUBSET:
    generator = torch.Generator().manual_seed(SEED)
    transfer_train_indices = torch.randperm(len(transfer_train_full), generator=generator)[:TRANSFER_TRAIN_SUBSET_SIZE]
    transfer_test_indices = torch.randperm(len(transfer_test_full), generator=generator)[:TRANSFER_TEST_SUBSET_SIZE]
    transfer_train_source = Subset(transfer_train_full, transfer_train_indices)
    transfer_test_dataset = Subset(transfer_test_full, transfer_test_indices)
else:
    transfer_train_source = transfer_train_full
    transfer_test_dataset = transfer_test_full

transfer_val_size = int(0.2 * len(transfer_train_source))
transfer_train_size = len(transfer_train_source) - transfer_val_size

transfer_train_dataset, transfer_val_dataset = random_split(
    transfer_train_source,
    [transfer_train_size, transfer_val_size],
    generator=torch.Generator().manual_seed(SEED),
)

transfer_train_loader = DataLoader(transfer_train_dataset, batch_size=TRANSFER_BATCH_SIZE, shuffle=True)
transfer_val_loader = DataLoader(transfer_val_dataset, batch_size=TRANSFER_BATCH_SIZE, shuffle=False)
transfer_test_loader = DataLoader(transfer_test_dataset, batch_size=TRANSFER_BATCH_SIZE, shuffle=False)

print("Transfer train:", len(transfer_train_dataset))
print("Transfer validation:", len(transfer_val_dataset))
print("Transfer test:", len(transfer_test_dataset))

## 21. Creazione del modello pre-addestrato

La cella sotto prova a caricare i pesi pre-addestrati. Se l'ambiente non ha connessione internet e i pesi non sono gia in cache, viene creato comunque un modello ResNet18 non pre-addestrato, cosi il notebook resta eseguibile. Per fare vero transfer learning servono pero i pesi pre-addestrati.

In [ ]:
def build_resnet18_transfer_model(num_classes=10, freeze_features=True):
    try:
        model_transfer = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        print("ResNet18 caricata con pesi pre-addestrati ImageNet.")
    except Exception as error:
        print("Non e stato possibile scaricare/caricare i pesi pre-addestrati.")
        print("Creo una ResNet18 senza pesi pre-addestrati. Errore:", error)
        model_transfer = models.resnet18(weights=None)

    if freeze_features:
        for parameter in model_transfer.parameters():
            parameter.requires_grad = False

    in_features = model_transfer.fc.in_features
    model_transfer.fc = nn.Linear(in_features, num_classes)
    return model_transfer

transfer_model = build_resnet18_transfer_model(num_classes=10, freeze_features=True).to(device)
print(transfer_model.fc)
print("Parametri totali:", sum(p.numel() for p in transfer_model.parameters()))
print("Parametri allenabili:", sum(p.numel() for p in transfer_model.parameters() if p.requires_grad))

In [ ]:
history_transfer = train_model(
    transfer_model,
    transfer_train_loader,
    transfer_val_loader,
    epochs=5,
    lr=0.001,
    weight_decay=1e-4,
)

In [ ]:
plot_loss(history_transfer, "Loss per epoca - Transfer learning ResNet18")
plot_accuracy(history_transfer, "Accuracy per epoca - Transfer learning ResNet18")

In [ ]:
transfer_test_accuracy, transfer_y_test_true, transfer_y_test_pred, transfer_y_test_proba = evaluate_model(transfer_model, transfer_test_loader, task="multiclass")
print(f"Accuracy test transfer learning: {transfer_test_accuracy:.3f}")
print("\nClassification report transfer learning:")
print(classification_report(transfer_y_test_true, transfer_y_test_pred, target_names=cifar_class_names, digits=3))

In [ ]:
cm_transfer = confusion_matrix(transfer_y_test_true, transfer_y_test_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm_transfer, cmap="Oranges")
plt.title("Confusion matrix - Transfer learning ResNet18")
plt.xlabel("Predetto")
plt.ylabel("Reale")
plt.xticks(range(10), cifar_class_names, rotation=45, ha="right")
plt.yticks(range(10), cifar_class_names)

for i in range(cm_transfer.shape[0]):
    for j in range(cm_transfer.shape[1]):
        color = "white" if cm_transfer[i, j] > cm_transfer.max() * 0.6 else "black"
        plt.text(j, i, cm_transfer[i, j], ha="center", va="center", color=color, fontsize=8)

plt.colorbar()
plt.show()

## 22. Confronto CNN da zero vs transfer learning

Confrontiamo la CNN allenata da zero su CIFAR-10 con la ResNet18 usata in transfer learning.

Attenzione: nella configurazione didattica usiamo subset e resize, quindi i risultati possono variare. Il punto principale e capire il workflow.

In [ ]:
color_comparison = pd.DataFrame({
    "modello": ["CNN CIFAR-10 da zero", "ResNet18 transfer learning"],
    "final_train_loss": [history_cifar_cnn["train_loss"][-1], history_transfer["train_loss"][-1]],
    "final_val_loss": [history_cifar_cnn["val_loss"][-1], history_transfer["val_loss"][-1]],
    "final_train_accuracy": [history_cifar_cnn["train_accuracy"][-1], history_transfer["train_accuracy"][-1]],
    "final_val_accuracy": [history_cifar_cnn["val_accuracy"][-1], history_transfer["val_accuracy"][-1]],
    "test_accuracy": [cifar_cnn_test_accuracy, transfer_test_accuracy],
    "parametri_allenabili": [
        sum(p.numel() for p in cifar_cnn_model.parameters() if p.requires_grad),
        sum(p.numel() for p in transfer_model.parameters() if p.requires_grad),
    ],
})

color_comparison

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_cifar_cnn["val_loss"], label="CNN da zero - validation loss")
plt.plot(history_transfer["val_loss"], label="Transfer learning - validation loss")
plt.title("Confronto validation loss - CIFAR-10")
plt.xlabel("Epoca")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_cifar_cnn["val_accuracy"], label="CNN da zero - validation accuracy")
plt.plot(history_transfer["val_accuracy"], label="Transfer learning - validation accuracy")
plt.title("Confronto validation accuracy - CIFAR-10")
plt.xlabel("Epoca")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 23. Cosa abbiamo imparato da Parte C e Parte D

- Le immagini RGB hanno 3 canali, quindi il primo layer convoluzionale deve usare `in_channels=3`.
- CIFAR-10 e piu difficile di Fashion-MNIST: immagini piu varie, oggetti piu complessi e sfondi colorati.
- Una CNN da zero impara feature partendo solo dal dataset disponibile.
- Nel transfer learning riusiamo feature gia apprese su un dataset grande.
- Congelare i layer iniziali riduce il numero di parametri allenabili e rende il training piu leggero.
- Se abbiamo abbastanza tempo e dati, possiamo anche fare fine-tuning sbloccando alcuni layer finali.

## 24. Esercizi suggeriti per il corso

1. Cambiare il numero di epoche e osservare training loss e validation loss.
2. Cambiare il numero di neuroni negli hidden layer della ANN.
3. Cambiare il numero di filtri della CNN Fashion-MNIST.
4. Cambiare il numero di filtri della CNN CIFAR-10.
5. Impostare `USE_CIFAR_SUBSET = False` e addestrare su tutto CIFAR-10.
6. Nel transfer learning, provare `freeze_features=False` e confrontare training time e accuracy.
7. Aggiungere data augmentation a CIFAR-10, per esempio `RandomHorizontalFlip` e `RandomCrop`.
8. Confrontare tutti i modelli usando validation loss, test accuracy e confusion matrix.


## 25. Mini riepilogo

- Fashion-MNIST contiene immagini `28 x 28` in scala di grigi.
- CIFAR-10 contiene immagini `32 x 32` a colori con 3 canali RGB.
- Una ANN classica richiede il flatten dell'immagine in un vettore.
- Una CNN usa convoluzioni e pooling per sfruttare la struttura spaziale dell'immagine.
- Il transfer learning riusa feature apprese da modelli pre-addestrati su dataset grandi.
- `CrossEntropyLoss` e la loss standard per classificazione multiclasse in PyTorch.
- Training loss e validation loss aiutano a capire apprendimento e overfitting.
